# Extension 1: Churn Prediction

**Goal:** Predict which active users will stop using the app within 30 days.

**Model:** `GBTClassifier` via Spark MLlib `Pipeline`

**Data sources:**
| Table | Role |
|---|---|
| `session_metrics` | Per-session behavioral features |
| `user_metadata` | User profile features |
| `daily_active_users` | Date range reference |

**Feasibility signal:** AUC-ROC > 0.70 confirms meaningful churn signal in existing features.

---
**Prerequisites:** Run `make run-jobs` (or `make quickstart`) before opening this notebook.
All 13 PostgreSQL tables must be populated.

## Cell 1 — Setup

In [ ]:
import os
import pandas as pd
import psycopg2
from pyspark.sql import SparkSession

PG = dict(
    host=os.getenv("POSTGRES_HOST", "postgres"),
    port=int(os.getenv("POSTGRES_PORT", 5432)),
    dbname=os.getenv("POSTGRES_DB", "analytics"),
    user=os.getenv("POSTGRES_USER", "analytics_user"),
    password=os.getenv("POSTGRES_PASSWORD", "analytics_pass"),
)

def pg_query(sql: str) -> pd.DataFrame:
    """Execute SQL and return a pandas DataFrame."""
    # psycopg2's context manager only manages transactions (commit/rollback),
    # NOT the connection lifecycle — close() must be called explicitly.
    conn = psycopg2.connect(**PG)
    try:
        return pd.read_sql(sql, conn)
    finally:
        conn.close()

# Connect to the existing Spark cluster.
# spark.driver.host must be set to the Jupyter container hostname so that
# spark-worker-1 and spark-worker-2 can open the callback channel to the driver.
spark = (
    SparkSession.builder
    .appName("ML Feasibility — Churn Prediction")
    .master(os.getenv("SPARK_MASTER_URL", "spark://spark-master:7077"))
    .config("spark.driver.host", "goodnote-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "master:", spark.sparkContext.master)

## Cell 2 — Feature Engineering

Aggregate per-user session behavior and join with profile metadata.
The churn label is derived from recency: users inactive for > 30 days are labelled churned.

In [ ]:
df = pg_query("""
    SELECT
        s.user_id,
        COUNT(*)                                   AS total_sessions,
        AVG(s.session_duration_ms) / 1000.0        AS avg_session_sec,
        AVG(s.actions_count)                       AS avg_actions_per_session,
        SUM(s.is_bounce)::float / COUNT(*)         AS personal_bounce_rate,
        MAX(s.session_start_time)                  AS last_seen,
        m.device_type,
        m.country,
        m.subscription_type
    FROM session_metrics s
    JOIN user_metadata m USING (user_id)
    GROUP BY s.user_id, m.device_type, m.country, m.subscription_type
""")

print(f"Users loaded: {len(df):,}")
df.head()

## Cell 3 — Label Definition

A user is **churned** if their last seen session was more than 30 days before the most recent
session date in the dataset. This is a proxy label derived entirely from existing data.

In [ ]:
CHURN_WINDOW_DAYS = 30

df["last_seen"] = pd.to_datetime(df["last_seen"])
reference_date = df["last_seen"].max()

df["churned"] = (
    (reference_date - df["last_seen"]).dt.days > CHURN_WINDOW_DAYS
).astype(int)

churn_rate = df["churned"].mean()
print(f"Reference date : {reference_date.date()}")
print(f"Churned users  : {df['churned'].sum():,}  ({churn_rate:.1%} of total)")
print(f"Retained users : {(~df['churned'].astype(bool)).sum():,}")

# NOTE: users last active within CHURN_WINDOW_DAYS of the reference date are
# structurally forced into "retained" — their recency alone prevents a churn
# label regardless of actual future behaviour. This is a proxy label; interpret
# AUC results with that in mind.
days_since_last_seen = (reference_date - df["last_seen"]).dt.days
print(f"\nUsers last active within {CHURN_WINDOW_DAYS}d of reference date (forced retained): "
      f"{(days_since_last_seen <= CHURN_WINDOW_DAYS).sum():,}")

df[["total_sessions", "avg_session_sec", "personal_bounce_rate", "churned"]].describe()

## Cell 4 — MLlib Pipeline

Pipeline stages:
1. `StringIndexer` — encode categorical features
2. `VectorAssembler` — combine numeric + encoded categoricals
3. `StandardScaler` — normalise features
4. `GBTClassifier` — gradient-boosted tree classifier

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Convert pandas → Spark (fill NaN before conversion)
sdf = spark.createDataFrame(
    df[["total_sessions", "avg_session_sec", "avg_actions_per_session",
        "personal_bounce_rate", "device_type", "country",
        "subscription_type", "churned"]]
    .fillna({"avg_session_sec": 0.0, "avg_actions_per_session": 0.0, "personal_bounce_rate": 0.0})
)
# Cache before split so the DataFrame is not recomputed for count() + fit()
sdf.cache()

# Encode categoricals
device_idx  = StringIndexer(inputCol="device_type",       outputCol="device_idx",  handleInvalid="keep")
country_idx = StringIndexer(inputCol="country",           outputCol="country_idx", handleInvalid="keep")
sub_idx     = StringIndexer(inputCol="subscription_type", outputCol="sub_idx",     handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=[
        "total_sessions", "avg_session_sec", "avg_actions_per_session",
        "personal_bounce_rate", "device_idx", "country_idx", "sub_idx",
    ],
    outputCol="features",
)

# StandardScaler removed: GBTClassifier is a tree-based model and is invariant
# to feature scaling. Scaling added cost and forced sparse→dense conversion.
gbt = GBTClassifier(
    labelCol="churned",
    featuresCol="features",
    maxIter=50,
    maxDepth=5,
    seed=42,
)

pipeline = Pipeline(stages=[device_idx, country_idx, sub_idx, assembler, gbt])

train, test = sdf.randomSplit([0.8, 0.2], seed=42)
print(f"Train rows: {train.count():,}   Test rows: {test.count():,}")
print("Fitting pipeline...")
model = pipeline.fit(train)
print("Done.")

## Cell 5 — Evaluation

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

predictions = model.transform(test)

auc_evaluator = BinaryClassificationEvaluator(labelCol="churned", metricName="areaUnderROC")
pr_evaluator  = BinaryClassificationEvaluator(labelCol="churned", metricName="areaUnderPR")
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="churned", predictionCol="prediction", metricName="accuracy"
)

auc = auc_evaluator.evaluate(predictions)
apr = pr_evaluator.evaluate(predictions)
acc = acc_evaluator.evaluate(predictions)

print(f"AUC-ROC  : {auc:.4f}   (feasibility threshold: 0.70)")
print(f"AUC-PR   : {apr:.4f}")
print(f"Accuracy : {acc:.4f}")
print()
if auc >= 0.70:
    print("RESULT: FEASIBLE — session + metadata features carry meaningful churn signal.")
    print("Recommended next step: promote to src/jobs/05_ml_churn.py")
else:
    print("RESULT: Further feature engineering needed before productionising.")

## Cell 6 — Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Derive feature names directly from the fitted assembler (stage index 3)
# so this cell never silently misaligns when inputCols change in Cell 4.
feature_names = model.stages[3].getInputCols()

# GBTClassifier is the last stage
importances = model.stages[-1].featureImportances.toArray()

sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(importances)), importances[sorted_idx])
ax.set_xticks(range(len(importances)))
ax.set_xticklabels([feature_names[i] for i in sorted_idx], rotation=35, ha="right")
ax.set_title("GBT Feature Importances — Churn Prediction")
ax.set_ylabel("Importance")
plt.tight_layout()
plt.show()

## Cell 7 — Confusion Matrix

In [ ]:
cm = (
    predictions
    .groupBy("churned", "prediction")
    .count()
    .orderBy("churned", "prediction")
    .toPandas()
)
print("Confusion matrix (churned vs prediction):")
print(cm.to_string(index=False))

## Cell 8 — Cleanup

In [ ]:
spark.stop()
print("Spark session stopped.")